# LLM Summarization — API Calls & Generation
**NLP Final Project — Dialectal Robustness**

This notebook handles all API calls and summary generation for both experimental tasks:

- **Task 1 – Generation Bias**: Unconstrained summarization; measure how often the model switches to American English.
- **Task 2 – Evaluation Bias**: Dialect-constrained summarization (explicitly US / UK / AUS); evaluated downstream with BERTScore and LLM-as-a-Judge.

Results are saved as JSONL checkpoints (resumable) and flat CSVs for analysis.

In [ ]:
# %pip install -U google-genai openai bert-score pandas tqdm
# Run the line above (uncommented) once to install required packages.
# OpenRouter is OpenAI-compatible — the `openai` package is all you need.

In [ ]:
import json
import os
import re
import time
import textwrap
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import pandas as pd
from tqdm.notebook import tqdm

from openai import OpenAI
from google import genai
from google.genai import types as genai_types

from spelling_markers import load_spelling_markers, compute_lsr, strip_quotes
from dialect_classifier import OUTLET_TO_DIALECT, _load_individual_file

## Configuration
Fill in your API keys below before running any other cells.

In [ ]:
# ── API Keys ─────────────────────────────────────────────────────────────
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"   # aistudio.google.com/app/apikey
GROQ_API_KEY   = "YOUR_GROQ_API_KEY"    # console.groq.com/keys

# ── Models ───────────────────────────────────────────────────────────────
GEMINI_MODEL = "gemini-2.5-flash"                                       # closed-source
LLAMA_MODEL  = "meta-llama/llama-4-scout-17b-16e-instruct"              # open-source via Groq (free)

# ── Paths ────────────────────────────────────────────────────────────────
ARTICLES_DIR  = "./summary_articles"   # swap back to "./articles" for the full run
SUMMARIES_DIR = "./summaries"
Path(SUMMARIES_DIR).mkdir(exist_ok=True)

# ── Run settings ─────────────────────────────────────────────────────────
MAX_ARTICLES    = 1     # 1 article per model for testing; set to None for full run
REQUEST_DELAY   = 1.5  # seconds between API calls
MAX_TOKENS      = 512
TEMPERATURE     = 0.3

print("Config loaded.")
print(f"  Articles dir  : {ARTICLES_DIR}")
print(f"  Summaries dir : {SUMMARIES_DIR}")
print(f"  Max articles  : {MAX_ARTICLES or 'all'}")

## Build `summary_articles/` — 2 Articles per Topic × Dialect

Selects the 2 best articles from each `articles/<topic>/<source>/` folder and copies them to `summary_articles/`.  
**Run this once** after `get_news.ipynb` has fetched articles. Re-running is safe — existing files are overwritten.

In [ ]:
import shutil

ARTICLES_SOURCE  = "./articles"         # written by get_news.ipynb
SAMPLE_PER_PAIR  = 2                    # articles per (topic × source)
SUMMARY_ARTICLES = "./summary_articles" # where make_summaries reads from


def _pick_best(folder: Path, n: int) -> list[Path]:
    """
    Return up to n article paths from folder.
    Articles with scraped full_text are ranked first so the LLM gets
    the most complete input possible.
    """
    with_full, without = [], []
    for f in sorted(folder.glob("article_*.json")):
        try:
            data = json.loads(f.read_text(encoding="utf-8"))
            (with_full if data.get("full_text") else without).append(f)
        except Exception:
            without.append(f)
    return (with_full + without)[:n]


def build_summary_articles(
    source_dir: str = ARTICLES_SOURCE,
    dest_dir:   str = SUMMARY_ARTICLES,
    n:          int = SAMPLE_PER_PAIR,
) -> None:
    src  = Path(source_dir)
    dest = Path(dest_dir)

    if not src.exists():
        print(f"Source directory '{source_dir}' not found. "
              "Run get_news.ipynb first.")
        return

    copied = skipped = 0
    report_rows = []

    # Walk articles/<topic>/<source>/
    for topic_dir in sorted(src.iterdir()):
        if not topic_dir.is_dir() or topic_dir.name in {"top_headlines"}:
            continue

        for source_dir_path in sorted(topic_dir.iterdir()):
            if not source_dir_path.is_dir():
                continue

            chosen = _pick_best(source_dir_path, n)

            if not chosen:
                skipped += 1
                report_rows.append({
                    "topic":   topic_dir.name,
                    "source":  source_dir_path.name,
                    "copied":  0,
                    "note":    "no articles found"
                })
                continue

            out_dir = dest / topic_dir.name / source_dir_path.name
            out_dir.mkdir(parents=True, exist_ok=True)

            for i, src_file in enumerate(chosen, start=1):
                shutil.copy2(src_file, out_dir / f"article_{i:03d}.json")
                copied += 1

            report_rows.append({
                "topic":  topic_dir.name,
                "source": source_dir_path.name,
                "copied": len(chosen),
                "note":   f"{sum(1 for f in chosen if json.loads(f.read_text()).get('full_text'))} with full text"
            })

    # Summary table
    df_report = pd.DataFrame(report_rows)
    if len(df_report):
        pivot = df_report.pivot_table(
            index="topic", columns="source", values="copied", aggfunc="sum"
        ).fillna(0).astype(int)
        print(f"Copied {copied} files into '{dest_dir}/'  "
              f"({skipped} (topic × source) pairs had no articles)\n")
        print(pivot.to_string())
    else:
        print("Nothing copied — check that articles/ contains JSON files.")


build_summary_articles()

## Initialise API Clients

In [ ]:
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# OpenRouter uses the OpenAI-compatible endpoint
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"Gemini       : {GEMINI_MODEL}  ✓")
print(f"LLaMA (free) : {LLAMA_MODEL} via OpenRouter  ✓")

In [ ]:
import requests as _req

print("=== GEMINI models supporting generateContent ===")
for m in gemini_client.models.list():
    supported = getattr(m, "supported_actions", None) or getattr(m, "supported_generation_methods", [])
    if "generateContent" in supported:
        print(f"  {m.name}")

print("\n=== OpenRouter free models (LLaMA / Mistral) ===")
resp = _req.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
    timeout=10,
)
if resp.status_code == 200:
    free_models = [
        m["id"] for m in resp.json().get("data", [])
        if m["id"].endswith(":free")
        and any(k in m["id"] for k in ("llama", "mistral", "qwen", "gemma"))
    ]
    for m in sorted(free_models):
        print(f"  {m}")
else:
    print(f"  Could not fetch model list ({resp.status_code})")

## Load Articles
Walks `./articles/` using the same loader as the dialect-classifier notebook.

In [71]:
def load_articles(articles_dir: str = ARTICLES_DIR) -> list[dict]:
    """
    Loads article JSON files from articles_dir.
    Handles both flat layout (files directly in the folder) and
    the nested layout used by get_news.ipynb (topic/outlet/article.json).
    """
    records = []

    for fpath in Path(articles_dir).rglob("*.json"):
        try:
            with open(fpath, encoding="utf-8") as f:
                raw = json.load(f)
        except (json.JSONDecodeError, OSError):
            continue

        # Resolve full text: prefer full_text field, fall back to snippet/description
        full_text = (
            raw.get("full_text")
            or raw.get("content_snippet")
            or raw.get("description")
            or ""
        ).strip()
        if not full_text:
            continue

        # Resolve outlet/dialect from the JSON's own 'source' field
        outlet  = raw.get("source", fpath.parent.name)
        dialect = OUTLET_TO_DIALECT.get(outlet)
        if dialect is None:
            # Try to infer from country code as fallback
            country_map = {"AU": "AUS", "GB": "UK", "US": "US"}
            dialect = country_map.get(raw.get("country", ""), "US")

        records.append({
            "full_text": full_text,
            "dialect":   dialect,
            "outlet":    outlet,
            "topic":     raw.get("topic", raw.get("topic_slug", fpath.parent.parent.name)),
            "title":     raw.get("title", ""),
            "file":      str(fpath),
        })

    return records


all_articles = load_articles()

if MAX_ARTICLES:
    all_articles = all_articles[:MAX_ARTICLES]

from collections import Counter
counts = Counter(a["outlet"] for a in all_articles)
print(f"Articles loaded : {len(all_articles)}")
for outlet, n in sorted(counts.items()):
    print(f"  {outlet:<12} {n} article(s)  (dialect: {OUTLET_TO_DIALECT.get(outlet, '?')})")
if all_articles:
    print(f"\nSample: [{all_articles[0]['dialect']}] {all_articles[0]['title'][:70]}")

Articles loaded : 1
  abc_au       1 article(s)  (dialect: AUS)

Sample: [AUS] Hikers lost near Kosciuszko found by artificial intelligence drone


## Prompt Templates

### Task 1 — Unconstrained
No dialect instruction; the model writes however it naturally defaults.

### Task 2 — Dialect-Constrained
Explicitly requests a specific dialect. We test all three (US / UK / AUS) for *every* article regardless of the article's original dialect.

In [ ]:
UNCONSTRAINED_PROMPT = (
    "Summarize the following news article in 3-5 sentences. "
    "Be factual and concise. Do not add information not in the article.\n\n"
    "Article:\n{text}\n\nSummary:"
)

DIALECT_INSTRUCTIONS = {
    "US":  "American English",
    "UK":  "British English",
    "AUS": "Australian English",
}

# Spelling cues per dialect — grounds the model in orthography, not stereotyped vocabulary
DIALECT_SPELLING_CUES = {
    "US":  "color, organize, realize, analyze, center, license, defense",
    "UK":  "colour, organise, realise, analyse, centre, licence, defence",
    "AUS": "colour, organise, realise, analyse, centre, licence, defence",
}

CONSTRAINED_PROMPT = (
    "Summarize the following news article in 3-5 sentences.\n\n"
    "Write as a professional journalist who naturally uses {dialect_label} spelling conventions "
    "(e.g. {spelling_cues}). Apply those spelling norms consistently throughout. "
    "Do NOT use stereotypical, archaic, or unusual regional vocabulary — "
    "the dialect difference should be subtle and reflect how a real {dialect_label} journalist would write, "
    "not a caricature.\n\n"
    "Article:\n{text}\n\nSummary:"
)

print("Prompt templates ready.")

## API Call Functions
`call_with_retry` wraps any API call with exponential back-off.  
`summarize_gemini` and `summarize_llama4` return the summary string or raise on final failure.

In [ ]:
def call_with_retry(fn, max_retries: int = 3, base_delay: float = 5.0):
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"    [retry {attempt+1}/{max_retries} in {delay:.0f}s] {exc}")
            time.sleep(delay)


def summarize_gemini(prompt: str) -> str:
    def _call():
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=genai_types.GenerateContentConfig(
                max_output_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            ),
        )
        return resp.text.strip()
    return call_with_retry(_call)


def summarize_llama(prompt: str) -> str:
    def _call():
        resp = openrouter_client.chat.completions.create(
            model=LLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
        )
        return resp.choices[0].message.content.strip()
    return call_with_retry(_call)


MODELS = {
    "gemini": summarize_gemini,
    "llama":  summarize_llama,   # free via OpenRouter
}

print("API functions ready:", list(MODELS))

## Checkpoint Helpers
Summaries are appended line-by-line to a JSONL file. If the cell is interrupted and re-run, already-completed (article, model) pairs are skipped automatically.

In [74]:
def append_record(record: dict, path: str) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: str) -> list[dict]:
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def done_keys(records: list[dict], key_fields: list[str]) -> set:
    return {tuple(r[k] for k in key_fields) for r in records}


print("Checkpoint helpers ready.")

Checkpoint helpers ready.


## Task 1 — Unconstrained Summarization (Generation Bias)

For every article, both models produce a free-form summary.  
The LSR (Lexical Switch Rate) computed downstream will measure how often each model switches to American English regardless of the source dialect.

In [75]:
TASK1_PATH = os.path.join(SUMMARIES_DIR, "task1_unconstrained.jsonl")

existing_t1 = load_jsonl(TASK1_PATH)
done_t1     = done_keys(existing_t1, ["file", "model"])
print(f"Task 1 — already done: {len(existing_t1)} records  ({len(done_t1)} unique (file, model) pairs)")

todo = [
    (art, model_name)
    for art in all_articles
    for model_name in MODELS
    if (art["file"], model_name) not in done_t1
]
print(f"Remaining        : {len(todo)}")

errors_t1 = []

for art, model_name in tqdm(todo, desc="Task 1"):
    prompt = UNCONSTRAINED_PROMPT.format(text=art["full_text"])
    try:
        summary = MODELS[model_name](prompt)
        record = {
            "task":            "unconstrained",
            "model":           model_name,
            "article_title":   art["title"],
            "topic":           art["topic"],
            "outlet":          art["outlet"],
            "article_dialect": art["dialect"],
            "file":            art["file"],
            "summary":         summary,
            "timestamp":       datetime.utcnow().isoformat(),
        }
        append_record(record, TASK1_PATH)
    except Exception as exc:
        errors_t1.append({"file": art["file"], "model": model_name, "error": str(exc)})
        print(f"  ERROR [{model_name}] {art['title'][:50]}: {exc}")
    time.sleep(REQUEST_DELAY)

t1_records = load_jsonl(TASK1_PATH)
print(f"\nTask 1 complete — {len(t1_records)} summaries saved to {TASK1_PATH}")
if errors_t1:
    print(f"  Errors: {len(errors_t1)} (see `errors_t1` list)")

Task 1 — already done: 0 records  (0 unique (file, model) pairs)
Remaining        : 2


Task 1:   0%|          | 0/2 [00:00<?, ?it/s]

    [retry 1/3  in 5s] 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 34.844204601s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
 

## Task 2 — Dialect-Constrained Summarization (Evaluation Bias)

Each article is summarized **three times per model** — once constrained to US, UK, and AUS English respectively.  
These summaries are later scored with BERTScore and LLM-as-a-Judge.

In [ ]:
TASK2_PATH = os.path.join(SUMMARIES_DIR, "task2_constrained.jsonl")

existing_t2 = load_jsonl(TASK2_PATH)
done_t2     = done_keys(existing_t2, ["file", "model", "target_dialect"])
print(f"Task 2 — already done: {len(existing_t2)} records")

todo2 = [
    (art, model_name, dialect)
    for art in all_articles
    for model_name in MODELS
    for dialect in DIALECT_INSTRUCTIONS
    if (art["file"], model_name, dialect) not in done_t2
]
print(f"Remaining        : {len(todo2)}")

errors_t2 = []

for art, model_name, dialect in tqdm(todo2, desc="Task 2"):
    prompt = CONSTRAINED_PROMPT.format(
        text=art["full_text"],
        dialect_label=DIALECT_INSTRUCTIONS[dialect],
        spelling_cues=DIALECT_SPELLING_CUES[dialect],
    )
    try:
        summary = MODELS[model_name](prompt)
        record = {
            "task":            "constrained",
            "model":           model_name,
            "article_title":   art["title"],
            "topic":           art["topic"],
            "outlet":          art["outlet"],
            "article_dialect": art["dialect"],
            "target_dialect":  dialect,
            "file":            art["file"],
            "summary":         summary,
            "timestamp":       datetime.utcnow().isoformat(),
        }
        append_record(record, TASK2_PATH)
    except Exception as exc:
        errors_t2.append({"file": art["file"], "model": model_name,
                          "dialect": dialect, "error": str(exc)})
        print(f"  ERROR [{model_name}/{dialect}] {art['title'][:45]}: {exc}")
    time.sleep(REQUEST_DELAY)

t2_records = load_jsonl(TASK2_PATH)
print(f"\nTask 2 complete — {len(t2_records)} summaries saved to {TASK2_PATH}")
if errors_t2:
    print(f"  Errors: {len(errors_t2)} (see `errors_t2` list)")

## Export to CSV
Produces two flat CSVs that are easier to work with in downstream analysis notebooks.

In [77]:
df_t1 = pd.DataFrame(load_jsonl(TASK1_PATH))
df_t2 = pd.DataFrame(load_jsonl(TASK2_PATH))

csv_t1 = os.path.join(SUMMARIES_DIR, "task1_unconstrained.csv")
csv_t2 = os.path.join(SUMMARIES_DIR, "task2_constrained.csv")

df_t1.to_csv(csv_t1, index=False, encoding="utf-8")
df_t2.to_csv(csv_t2, index=False, encoding="utf-8")

print(f"Task 1 CSV : {csv_t1}  ({len(df_t1)} rows)")
print(f"Task 2 CSV : {csv_t2}  ({len(df_t2)} rows)")
print()
if len(df_t1):
    print("Task 1 breakdown (model × article_dialect):")
    print(df_t1.groupby(["model", "article_dialect"])["summary"].count().unstack(fill_value=0))
if len(df_t2):
    print("\nTask 2 breakdown (model × target_dialect):")
    print(df_t2.groupby(["model", "target_dialect"])["summary"].count().unstack(fill_value=0))

Task 1 CSV : ./summaries/task1_unconstrained.csv  (0 rows)
Task 2 CSV : ./summaries/task2_constrained.csv  (0 rows)



## LSR on Generated Summaries (Task 1)
Applies the Lexical Switch Rate from `spelling_markers.py` to each unconstrained summary.  
A high LSR on a UK or AUS article indicates the model "drifted" to American English — the core Generation Bias signal.

In [78]:
markers = load_spelling_markers()
print(f"Markers — US: {len(markers['US']):,}  UK: {len(markers['UK']):,}  AU: {len(markers['AU']):,}")


def compute_summary_lsr(summary: str) -> dict:
    clean = strip_quotes(summary)
    return compute_lsr(clean, markers)


df_t1 = pd.DataFrame(load_jsonl(TASK1_PATH))

if len(df_t1):
    lsr_results = df_t1["summary"].apply(compute_summary_lsr)
    df_t1["lsr"]                 = lsr_results.apply(lambda r: r["lsr"])
    df_t1["total_dialect_tokens"] = lsr_results.apply(lambda r: r["total_dialect_tokens"])
    df_t1["us_tokens"]            = lsr_results.apply(lambda r: r["us_tokens"])

    lsr_pivot = (
        df_t1[df_t1["total_dialect_tokens"] > 0]
        .groupby(["model", "article_dialect"])["lsr"]
        .agg(["mean", "std", "count"])
        .round(4)
    )
    print("Mean LSR by model and source dialect (higher = more American English):")
    print(lsr_pivot.to_string())

    df_t1.to_csv(csv_t1, index=False, encoding="utf-8")
    print(f"\nLSR columns added and saved to {csv_t1}")
else:
    print("No Task 1 summaries found — run the Task 1 cell first.")

Markers — US: 2,766  UK: 1,730  AU: 2,786
No Task 1 summaries found — run the Task 1 cell first.


## LSR on Constrained Summaries (Task 2)
Validates constraint adherence: a constrained-US summary should have LSR ≈ 1.0; constrained-UK/AUS should have LSR ≈ 0.0.

In [79]:
df_t2 = pd.DataFrame(load_jsonl(TASK2_PATH))

if len(df_t2):
    lsr2 = df_t2["summary"].apply(compute_summary_lsr)
    df_t2["lsr"]                 = lsr2.apply(lambda r: r["lsr"])
    df_t2["total_dialect_tokens"] = lsr2.apply(lambda r: r["total_dialect_tokens"])

    lsr2_pivot = (
        df_t2[df_t2["total_dialect_tokens"] > 0]
        .groupby(["model", "target_dialect"])["lsr"]
        .mean()
        .unstack(fill_value=float("nan"))
        .round(4)
    )
    print("Mean LSR per model × target dialect (US col should be ~1.0, UK/AUS ~0.0):")
    print(lsr2_pivot.to_string())

    df_t2.to_csv(csv_t2, index=False, encoding="utf-8")
    print(f"\nLSR columns added and saved to {csv_t2}")
else:
    print("No Task 2 summaries found — run the Task 2 cell first.")

No Task 2 summaries found — run the Task 2 cell first.


## LLM-as-a-Judge — Quality Scoring (Task 2)

Each constrained summary is scored 1–10 by the judge model (Gemini by default).  
The score reflects summary quality relative to the source article — bias will appear as a systematic quality gap between dialects.

> **Note:** The judge itself may have American English bias, which is exactly the confound the proposal acknowledges. Use in combination with BERTScore for a fuller picture.

In [80]:
JUDGE_MODEL    = "gemini"   # change to "llama4" to use an open-source judge
JUDGE_PATH     = os.path.join(SUMMARIES_DIR, "task2_judge_scores.jsonl")
JUDGE_DELAY    = 1.5        # seconds between judge calls

JUDGE_PROMPT = (
    "You are an expert evaluator of news article summaries.\n\n"
    "Rate the following summary on a scale from 1 to 10, where:\n"
    "  10 = perfect: accurate, complete, concise, no hallucinations\n"
    "   7 = good: mostly accurate, minor omissions\n"
    "   4 = fair: captures main idea but notable gaps or inaccuracies\n"
    "   1 = poor: inaccurate, incoherent, or off-topic\n\n"
    "Original article:\n{article}\n\n"
    "Summary to evaluate:\n{summary}\n\n"
    "Respond ONLY with valid JSON on a single line, like:\n"
    '{{"score": <1-10>, "reasoning": "<one sentence>"}}'
)


def judge_summary(article_text: str, summary: str) -> dict:
    prompt = JUDGE_PROMPT.format(article=article_text[:3000], summary=summary)
    raw = MODELS[JUDGE_MODEL](prompt)
    raw = raw.strip().lstrip("```json").rstrip("`").strip()
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        raise ValueError(f"Could not parse judge response: {raw[:120]}")
    return json.loads(match.group())


existing_judge = load_jsonl(JUDGE_PATH)
done_judge     = done_keys(existing_judge, ["file", "model", "target_dialect"])
df_t2_all      = pd.DataFrame(load_jsonl(TASK2_PATH))

todo_judge = [
    row for _, row in df_t2_all.iterrows()
    if (row["file"], row["model"], row["target_dialect"]) not in done_judge
]
print(f"Judge calls remaining: {len(todo_judge)} / {len(df_t2_all)}")

judge_errors = []

for row in tqdm(todo_judge, desc="LLM-Judge"):
    try:
        with open(row["file"], encoding="utf-8") as f:
            raw_art = json.load(f)
        article_text = _load_individual_file(row["file"]).strip()
        result = judge_summary(article_text, row["summary"])
        record = {
            "file":           row["file"],
            "model":          row["model"],
            "target_dialect": row["target_dialect"],
            "article_dialect":row["article_dialect"],
            "topic":          row["topic"],
            "score":          result.get("score"),
            "reasoning":      result.get("reasoning", ""),
            "judge_model":    JUDGE_MODEL,
            "timestamp":      datetime.utcnow().isoformat(),
        }
        append_record(record, JUDGE_PATH)
    except Exception as exc:
        judge_errors.append({"file": row["file"], "error": str(exc)})
        print(f"  ERROR: {exc}")
    time.sleep(JUDGE_DELAY)

df_judge = pd.DataFrame(load_jsonl(JUDGE_PATH))
if len(df_judge):
    judge_csv = os.path.join(SUMMARIES_DIR, "task2_judge_scores.csv")
    df_judge.to_csv(judge_csv, index=False)
    print(f"\nScores saved: {judge_csv}")
    print(df_judge.groupby(["model", "target_dialect"])["score"]
          .mean().unstack(fill_value=float("nan")).round(2).to_string())
if judge_errors:
    print(f"Judge errors: {len(judge_errors)}")

Judge calls remaining: 0 / 0


LLM-Judge: 0it [00:00, ?it/s]

## BERTScore — Semantic Overlap with Source Article (Task 2)

BERTScore measures semantic similarity between the generated summary (hypothesis) and the original article (reference).  
Lower F1 for a given dialect → that dialect's summaries are semantically further from the source.

In [81]:
from bert_score import score as bert_score

BERTSCORE_PATH = os.path.join(SUMMARIES_DIR, "task2_bertscore.jsonl")
BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"  # strong BERTScore backbone
BERTSCORE_LANG  = "en"

df_t2_bs = pd.DataFrame(load_jsonl(TASK2_PATH))
done_bs  = done_keys(load_jsonl(BERTSCORE_PATH), ["file", "model", "target_dialect"])

todo_bs = [
    row for _, row in df_t2_bs.iterrows()
    if (row["file"], row["model"], row["target_dialect"]) not in done_bs
]
print(f"BERTScore pairs remaining: {len(todo_bs)} / {len(df_t2_bs)}")

# Batch in chunks to avoid OOM
BATCH_SIZE = 32

def load_text(fpath):
    return _load_individual_file(fpath).strip()

for batch_start in tqdm(range(0, len(todo_bs), BATCH_SIZE), desc="BERTScore"):
    batch  = todo_bs[batch_start : batch_start + BATCH_SIZE]
    hyps   = [row["summary"]      for row in batch]
    refs   = [load_text(row["file"]) for row in batch]

    P, R, F1 = bert_score(
        hyps, refs,
        model_type=BERTSCORE_MODEL,
        lang=BERTSCORE_LANG,
        verbose=False,
    )

    for row, p, r, f in zip(batch, P.tolist(), R.tolist(), F1.tolist()):
        record = {
            "file":            row["file"],
            "model":           row["model"],
            "target_dialect":  row["target_dialect"],
            "article_dialect": row["article_dialect"],
            "topic":           row["topic"],
            "precision":       round(p, 4),
            "recall":          round(r, 4),
            "f1":              round(f, 4),
            "bertscore_model": BERTSCORE_MODEL,
        }
        append_record(record, BERTSCORE_PATH)

df_bs = pd.DataFrame(load_jsonl(BERTSCORE_PATH))
if len(df_bs):
    bs_csv = os.path.join(SUMMARIES_DIR, "task2_bertscore.csv")
    df_bs.to_csv(bs_csv, index=False)
    print(f"\nBERTScore results saved: {bs_csv}")
    print("Mean F1 by model × target dialect:")
    print(df_bs.groupby(["model", "target_dialect"])["f1"]
          .mean().unstack(fill_value=float("nan")).round(4).to_string())

BERTScore pairs remaining: 0 / 0


BERTScore: 0it [00:00, ?it/s]

## Sanity Check — Sample Output
Prints one article with its unconstrained summaries from both models side by side.

In [83]:
df_t1_check = pd.DataFrame(load_jsonl(TASK1_PATH))

if len(df_t1_check) == 0:
    print("No summaries yet — run Task 1 first.")
else:
    sample_title = df_t1_check.iloc[0]["article_title"]
    sample_rows  = df_t1_check[df_t1_check["article_title"] == sample_title]

    print(f"Article : {sample_title}")
    print(f"Topic   : {sample_rows.iloc[0]['topic']}")
    print(f"Outlet  : {sample_rows.iloc[0]['outlet']}  "
          f"(dialect: {sample_rows.iloc[0]['article_dialect']})")
    print()
    for _, row in sample_rows.iterrows():
        print(f"─── {row['model'].upper()} ───")
        wrapped = textwrap.fill(row["summary"], width=90, subsequent_indent="  ")
        print(wrapped)
        print()

No summaries yet — run Task 1 first.
